# P10c — the anti-prior ON arm, on a free Kaggle T4.

The Mac leg of this pair is abandoned (thermal budget): `antiprior_off_s0` ran to
3,314 of 3,600 updates on an M5 and its own `--stop-factor 3.0` guard ended it on a
596.40 s step against a 6.58 s median. Its weights are published as the
`antiprior_off_s0-weights` release, so this kernel has one job: produce the `on`
side at the pinned dose.

Nothing here hand-types `myna.train`. Every arm goes through `kaggle/run.py`, the
canonical launcher, so the two arms differ by one flag value. `--free-gib 9.0` is
kept rather than adapted: the recovered V1-B metrics record `mem_plan_free_gib = 9.0`
on a Tesla T4, so 9.0 IS that box's pin, and changing it would silently re-clamp the
batch — which is the dose, which is the figure.

Cell 1 refuses to continue without a GPU. A CPU session would still train, ~10x
slower, and print numbers that look identical — the worst possible failure.


In [ ]:
import glob, os, shutil, torch
print("torch", torch.__version__, "| cuda build", torch.version.cuda,
      "| device_count", torch.cuda.device_count())
print("ACCELERATOR_TYPE", repr(os.environ.get("ACCELERATOR_TYPE")),
      "| nvidia-smi at", repr(shutil.which("nvidia-smi")), "| /dev/nvidia*", glob.glob("/dev/nvidia*"))
if not torch.cuda.is_available():
    raise SystemExit(
        "NO GPU ATTACHED -- CPU session, refusing to train. The request is not the bug: on this "
        "account five variants (machine_shape NvidiaTeslaT4 and NvidiaL4, --accelerator "
        "NvidiaTeslaT4 and NVIDIA_TESLA_T4_X_2, docker_image_pinning_type latest with docker_image "
        "removed) all came back CPU, `kaggle kernels pull -m` shows the server itself stored "
        "enable_gpu=true and machine_shape=NvidiaTeslaT4, and `kaggle quota` prints GPU 0.00h used / "
        "30.00h remaining. The identical recipe trained V1-B on aashishkumarmahato01, whose log line "
        "120 reads `device: Tesla T4` with 14806 MiB free. So this account is denied the hardware at "
        "provisioning. SPEC §9.53(vi). Two closes, both browser-side: read the Accelerator dropdown "
        "for this account, or re-mint a token for aashishkumarmahato01.")

In [ ]:
!git clone -q https://github.com/aashish254/myna /kaggle/working/myna
!git -C /kaggle/working/myna checkout -q 52bc9a2665e0f6aaf904244c5c600ca8a16cbe61
!git -C /kaggle/working/myna log --oneline -1


In [ ]:
%cd /kaggle/working/myna
# Prove the tree that trains carries the ask() fix, instead of trusting a SHA. Fail loud:
# 5.6 h spent on a tree without it would print numbers that mean something else, and a grep
# that only *shows* the line is how a silent miss survives to the end of a run.
import re, subprocess
src = open("src/myna/engine.py").read()
hit = re.search(r"(@torch\.no_grad\(\)\s*\n\s*)?def ask\(", src)
if not hit or not hit.group(1):
    raise SystemExit("THE CLONED TREE LACKS THE ask() FIX: no @torch.no_grad() above "
                     "Observation.ask(). Check the checkout SHA before spending GPU hours.")
print("witness: @torch.no_grad() sits above def ask() · HEAD",
      subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -e . 2>&1 | tail -3
!python -c "import torch, tokenizers, pyarrow, numpy; print('imports ok', torch.__version__)"


In [ ]:
!python kaggle/run.py --corpus data/decision-v2-pilot --out-root /kaggle/working --device cuda --config v0 --steps 3600 --batch 10 --seed 0 --score-loss ce --stop-factor 3.0 --save-every 250 --free-gib 9.0 --name antiprior_on_s0 --anti-prior on --check


In [ ]:
# The exact command this kernel runs, printed by the launcher itself.
!python kaggle/run.py --corpus data/decision-v2-pilot --out-root /kaggle/working --device cuda --config v0 --steps 3600 --batch 10 --seed 0 --score-loss ce --stop-factor 3.0 --save-every 250 --free-gib 9.0 --name antiprior_on_s0 --anti-prior on --dry-run


In [ ]:
# 3,600 updates at V1-B's measured T4 rate (5.618 s/update) is ~5.6 h of the 9 h cap.
!python kaggle/run.py --corpus data/decision-v2-pilot --out-root /kaggle/working --device cuda --config v0 --steps 3600 --batch 10 --seed 0 --score-loss ce --stop-factor 3.0 --save-every 250 --free-gib 9.0 --name antiprior_on_s0 --anti-prior on 2>&1 | tee /kaggle/working/antiprior_on_s0.train.log


In [ ]:
!python -m myna.report --suite data/decision-v2-pilot --split test \
    --metrics /kaggle/working/antiprior_on_s0/metrics.json \
    --laya runs/laya_decision_v2_test.json \
    --out /kaggle/working/antiprior_on_s0.report.json 2>&1 | tail -20


In [ ]:
import json, hashlib, pathlib
d = pathlib.Path("/kaggle/working/antiprior_on_s0")
for f in ("model.pt","tokenizer.json","metrics.json"):
    p = d/f
    print(f, p.stat().st_size if p.exists() else "MISSING",
          hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.exists() else "")
m = json.load(open(d/"metrics.json"))
for k in ("last_step","steps_requested","stopped","paraphrase_draws","anti_prior",
          "batch","mem_plan_free_gib","temperature"):
    print(k, "=", m.get(k))
